<a href="https://colab.research.google.com/github/miriamamin1213-ux/Seed42_Models/blob/main/GPT2_Clinical%20Narrative.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score
)

from transformers import (
    GPT2Tokenizer,
    GPT2Model
)


# ============================================================
# Reproducibility
# ============================================================

def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.use_deterministic_algorithms(
        True,
        warn_only=True
    )

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False


SEED = 42
seed_everything(SEED)


# ============================================================
# Device
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)


# ============================================================
# Load and preprocess data
# ============================================================

import gdown
import pandas as pd

gdown.download(
    id="1DGDH2e6dEkpdwwtCvH6ds1CwJT2Hdmkv",
    output="HPV2025.xlsx",
    quiet=False
)

df = pd.read_excel("HPV2025.xlsx")

df = df.dropna(subset=["HPV Status"])

tobacco_mode = df["Tobacco Consumption"].mode()[0]
df["Tobacco Consumption"] = (
    df["Tobacco Consumption"].fillna(tobacco_mode)
)

alcohol_mode = df["Alcohol Consumption"].mode()[0]
df["Alcohol Consumption"] = (
    df["Alcohol Consumption"].fillna(alcohol_mode)
)

df = df.drop(
    columns=[
        "PatientID",
        "CenterID",
        "Task 1",
        "Task 2",
        "Task 3"
    ]
)

df = df.dropna()

print("Dataset shape:", df.shape)


# ============================================================
# Normalise stage labels for readable text
# ============================================================

def format_t_stage(value):
    value = str(value)

    if value.startswith("T"):
        return value

    return f"T{int(float(value))}"


def format_n_stage(value):
    value = str(value)

    if value.startswith("N"):
        return value

    return f"N{int(float(value))}"


def format_m_stage(value):
    value = str(value)

    if value.startswith("M"):
        return value

    return f"M{int(float(value))}"


# ============================================================
# Convert one patient record into text
# ============================================================

def patient_to_text(row):

    gender = "male" if row["Gender"] == 1 else "female"

    smoker = (
        "current smoker"
        if row["Tobacco Consumption"] == 1
        else "non-smoker"
    )

    alcohol = (
        "consumes alcohol"
        if row["Alcohol Consumption"] == 1
        else "does not consume alcohol"
    )

    relapse = (
        "has experienced disease relapse"
        if row["Relapse"] == 1
        else "has not experienced disease relapse"
    )

    text = (
        f"This patient is a {row['Age']:.0f}-year-old {gender}. "
        f"The patient is a {smoker} and {alcohol}. "
        f"Performance status is {row['Performance Status']}. "
        f"The patient {relapse}. "
        f"Relapse-free survival is {row['RFS']} days. "
        f"Treatment type is {row['Treatment']}. "
        f"Tumour stage is {format_t_stage(row['T-stage'])}. "
        f"Nodal stage is {format_n_stage(row['N-stage'])}. "
        f"Metastatic stage is {format_m_stage(row['M-stage'])}. "
        f"Predict whether the patient is HPV positive or HPV negative."
    )

    return text


df["patient_text"] = df.apply(
    patient_to_text,
    axis=1
)

print("\nExample patient text:")
print(df["patient_text"].iloc[0])


# ============================================================
# Text and labels
# ============================================================

texts = df["patient_text"].tolist()

labels = (
    df["HPV Status"]
    .astype(int)
    .to_numpy()
)


# ============================================================
# Train-test split
# ============================================================

train_texts, test_texts, y_train, y_test = (
    train_test_split(
        texts,
        labels,
        test_size=0.2,
        random_state=SEED,
        stratify=labels
    )
)

print("\nTrain class distribution:")
print(pd.Series(y_train).value_counts())

print("\nTest class distribution:")
print(pd.Series(y_test).value_counts())


# ============================================================
# Tokenizer
# ============================================================

MODEL_NAME = "openai-community/gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(
    MODEL_NAME
)

# GPT-2 has no default padding token
tokenizer.pad_token = tokenizer.eos_token

MAX_LENGTH = 128


# ============================================================
# Dataset
# ============================================================

class HPVTextDataset(Dataset):

    def __init__(
        self,
        texts,
        labels,
        tokenizer,
        max_length=128
    ):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):

        encoding = self.tokenizer(
            self.texts[index],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": (
                encoding["input_ids"].squeeze(0)
            ),
            "attention_mask": (
                encoding["attention_mask"].squeeze(0)
            ),
            "label": torch.tensor(
                self.labels[index],
                dtype=torch.long
            )
        }


train_dataset = HPVTextDataset(
    texts=train_texts,
    labels=y_train,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

test_dataset = HPVTextDataset(
    texts=test_texts,
    labels=y_test,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)


# ============================================================
# DataLoaders
# ============================================================

BATCH_SIZE = 256

train_generator = torch.Generator()
train_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_generator,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


# ============================================================
# GPT-2 text classifier
# ============================================================

class HPVNetGPT2Text(nn.Module):

    def __init__(
        self,
        model_name="openai-community/gpt2",
        num_classes=2,
        dropout=0.1,
        freeze_gpt2=True
    ):
        super().__init__()

        self.freeze_gpt2 = freeze_gpt2

        self.gpt2 = GPT2Model.from_pretrained(
            model_name
        )

        self.gpt2.config.use_cache = False
        self.gpt2.config.pad_token_id = (
            tokenizer.pad_token_id
        )

        hidden_size = self.gpt2.config.hidden_size

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

        if freeze_gpt2:
            self.gpt2.requires_grad_(False)

    def train(self, mode=True):
        super().train(mode)

        if self.freeze_gpt2:
            self.gpt2.eval()

        return self

    def forward(
        self,
        input_ids,
        attention_mask
    ):
        outputs = self.gpt2(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            return_dict=True
        )

        hidden_states = outputs.last_hidden_state

        # Masked mean pooling
        expanded_mask = (
            attention_mask
            .unsqueeze(-1)
            .expand_as(hidden_states)
            .float()
        )

        hidden_sum = (
            hidden_states * expanded_mask
        ).sum(dim=1)

        valid_token_count = (
            expanded_mask.sum(dim=1)
            .clamp(min=1e-9)
        )

        pooled_output = (
            hidden_sum / valid_token_count
        )

        logits = self.classifier(
            pooled_output
        )

        return logits


# ============================================================
# Model
# ============================================================

model = HPVNetGPT2Text(
    model_name=MODEL_NAME,
    num_classes=2,
    freeze_gpt2=True
).to(device)


# ============================================================
# Class weights
# ============================================================

class_counts = np.bincount(y_train)

class_weights = (
    len(y_train)
    / (
        len(class_counts) *
        class_counts
    )
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=device
)

print("\nClass weights:", class_weights)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)


# ============================================================
# Optimiser
# ============================================================

optimizer = torch.optim.AdamW(
    filter(
        lambda parameter: parameter.requires_grad,
        model.parameters()
    ),
    lr=1e-3,
    weight_decay=1e-4
)

print(
    "Trainable parameters:",
    sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )
)


# ============================================================
# Training
# ============================================================

EPOCHS = 100
best_val_loss = float("inf")
best_epoch = 0

for epoch in range(EPOCHS):

    model.train()

    train_loss_sum = 0.0
    train_sample_count = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        attention_mask = batch[
            "attention_mask"
        ].to(
            device,
            non_blocking=True
        )

        labels = batch["label"].to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        train_loss = criterion(
            outputs,
            labels
        )

        train_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            filter(
                lambda parameter: parameter.requires_grad,
                model.parameters()
            ),
            max_norm=1.0
        )

        optimizer.step()

        train_loss_sum += (
            train_loss.item() *
            input_ids.size(0)
        )

        train_sample_count += (
            input_ids.size(0)
        )

    average_train_loss = (
        train_loss_sum /
        train_sample_count
    )

    model.eval()

    test_loss_sum = 0.0
    test_sample_count = 0

    with torch.no_grad():

        for batch in test_loader:

            input_ids = batch[
                "input_ids"
            ].to(
                device,
                non_blocking=True
            )

            attention_mask = batch[
                "attention_mask"
            ].to(
                device,
                non_blocking=True
            )

            labels = batch["label"].to(
                device,
                non_blocking=True
            )

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            test_loss = criterion(
                outputs,
                labels
            )

            test_loss_sum += (
                test_loss.item() *
                input_ids.size(0)
            )

            test_sample_count += (
                input_ids.size(0)
            )

    average_test_loss = (
        test_loss_sum /
        test_sample_count
    )

    if average_test_loss < best_val_loss:

        best_val_loss = average_test_loss
        best_epoch = epoch + 1

        torch.save(
            model.state_dict(),
            "best_model_gpt2_text.pth"
        )

    if (epoch + 1) % 5 == 0:

        print(
            f"Epoch {epoch + 1:03d}, "
            f"Train={average_train_loss:.4f}, "
            f"Test={average_test_loss:.4f}, "
            f"Best Epoch={best_epoch}"
        )


print("\nBest Test Loss:", best_val_loss)
print("Best Epoch:", best_epoch)


# ============================================================
# Load best model
# ============================================================

checkpoint = torch.load(
    "best_model_gpt2_text.pth",
    map_location=device
)

model.load_state_dict(checkpoint)
model.eval()


# ============================================================
# Inference
# ============================================================

all_probabilities = []
all_predictions = []
all_targets = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        attention_mask = batch[
            "attention_mask"
        ].to(
            device,
            non_blocking=True
        )

        labels = batch["label"]

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        predictions = torch.argmax(
            probabilities,
            dim=1
        )

        all_probabilities.append(
            probabilities.cpu()
        )

        all_predictions.append(
            predictions.cpu()
        )

        all_targets.append(
            labels.cpu()

                    )


probabilities = torch.cat(
    all_probabilities,
    dim=0
).numpy()

predicted = torch.cat(
    all_predictions,
    dim=0
).numpy()

y_true = torch.cat(
    all_targets,
    dim=0
).numpy()


# ============================================================
# Evaluation
# ============================================================

results = classification_report(
    y_true,
    predicted,
    digits=4,
    zero_division=0
)

print("\nClassification report:")
print(results)

bal_acc = balanced_accuracy_score(
    y_true,
    predicted
)

f1 = f1_score(
    y_true,
    predicted,
    zero_division=0
)

auc = roc_auc_score(
    y_true,
    probabilities[:, 1]
)

print(f"Balanced Accuracy: {bal_acc:.4f}")
print(f"F1-score:          {f1:.4f}")
print(f"AUC:               {auc:.4f}")

Using device: cuda


Downloading...
From: https://drive.google.com/uc?id=1DGDH2e6dEkpdwwtCvH6ds1CwJT2Hdmkv
To: /content/HPV2025.xlsx
100%|██████████| 66.0k/66.0k [00:00<00:00, 21.9MB/s]


Dataset shape: (423, 12)

Example patient text:
This patient is a 76-year-old male. The patient is a current smoker and does not consume alcohol. Performance status is 1.0. The patient has not experienced disease relapse. Relapse-free survival is 1865.0 days. Treatment type is 1.0. Tumour stage is T3. Nodal stage is N0. Metastatic stage is M0. Predict whether the patient is HPV positive or HPV negative.

Train class distribution:
1    320
0     18
Name: count, dtype: int64

Test class distribution:
1    80
0     5
Name: count, dtype: int64


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Class weights: tensor([9.3889, 0.5281], device='cuda:0')
Trainable parameters: 100226
Epoch 005, Train=0.7237, Test=0.6905, Best Epoch=4
Epoch 010, Train=0.6846, Test=0.6936, Best Epoch=7
Epoch 015, Train=0.6815, Test=0.6813, Best Epoch=15
Epoch 020, Train=0.6779, Test=0.6936, Best Epoch=16
Epoch 025, Train=0.6884, Test=0.6742, Best Epoch=25
Epoch 030, Train=0.6775, Test=0.6791, Best Epoch=27
Epoch 035, Train=0.6778, Test=0.6638, Best Epoch=35
Epoch 040, Train=0.6697, Test=0.6588, Best Epoch=40
Epoch 045, Train=0.6510, Test=0.6694, Best Epoch=43
Epoch 050, Train=0.6826, Test=0.6396, Best Epoch=50
Epoch 055, Train=0.6760, Test=0.6308, Best Epoch=55
Epoch 060, Train=0.6820, Test=0.6372, Best Epoch=55
Epoch 065, Train=0.6292, Test=0.6266, Best Epoch=63
Epoch 070, Train=0.6482, Test=0.6013, Best Epoch=70
Epoch 075, Train=0.6164, Test=0.5882, Best Epoch=75
Epoch 080, Train=0.6175, Test=0.5976, Best Epoch=79
Epoch 085, Train=0.6307, Test=0.5603, Best Epoch=83
Epoch 090, Train=0.6335, Test=0